In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [ ]:
# PyTorch setup
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Availability: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)

In [ ]:
# Hyperparameters and constants
max_altitude = 10000.0
max_velocity = 500.0
c_g = 9.8085
c_m = 9.000
c_rho = 1.225
c_Cd = 0.461
c_A = 0.01558
w_data = 1.0
w_boundary = 1.0
w_physics = 1e-4
n_collocation = 1000000
n_boundary = 10000
n_epochs_adam = 3000
n_epochs_lbfgs = 10000
n_history_size_lbfgs = 50
epochs_print_rate = 100

In [ ]:
# Real flight data processing
def get_flight_data():
    df = pd.read_csv(os.path.join(os.path.abspath(''), "data", "taipan_fm2025.csv"))

    df = df[df['state'] > 1]
    df['h'] = df.apply(lambda x: float(x['h'].replace(',', '.')), axis=1)
    df['h'] = df['h'].rolling(window=5, center=True).mean()
    df['v'] = df['h'].diff() / (df['dt'] / 1e6)
    df['v'] = df['v'].rolling(window=15, center=True).mean()
    apogee = df['h'].max()

    df = df[['h', 'v']]
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    df = df.iloc[:1000]

    df['h'] = df['h'] / max_altitude
    df['v'] = df['v'] / max_velocity
    df['apogee'] = apogee / max_altitude

    h = torch.tensor(df['h'].values, dtype=torch.float32).unsqueeze(1)
    v = torch.tensor(df['v'].values, dtype=torch.float32).unsqueeze(1)
    y = torch.tensor(df['apogee'].values, dtype=torch.float32).unsqueeze(1)

    return h, v, y


# Boundary conditions: at apogee, velocity should be 0 + Normalization
def generate_boundary_points():
    h = torch.rand((n_boundary, 1))
    v = torch.zeros((n_boundary, 1))
    apogee = h

    return h, v, apogee


# Data generation for training the PINN + Normalization
def generate_collocation_points():
    h = torch.rand((n_collocation, 1))
    v = torch.rand((n_collocation, 1))

    return h, v

In [ ]:
# Define the Physics-Informed Neural Network (PINN) architecture for airbrake control
class AirbrakePINN(nn.Module):
    def __init__(self):
        super(AirbrakePINN, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )

    def forward(self, h, v):
        x = torch.cat([h, v], dim=1)
        return self.net(x)


def compute_acc(h, v):
    exponent = (-c_g * 0.02896 * h) / (8.314 * 288.15)
    dynamic_rho = 1.225 * torch.exp(exponent)
    return -c_g - (0.5 * dynamic_rho * v**2 * c_Cd * c_A) / c_m

In [ ]:
# Data preparation
h_data, v_data, y_data = get_flight_data()
h_boundary, v_boundary, y_boundary = generate_boundary_points()
h_colloc, v_colloc = generate_collocation_points()

h_data = h_data.to(device)
v_data = v_data.to(device)
y_data = y_data.to(device)
h_boundary = h_boundary.to(device)
v_boundary = v_boundary.to(device)
y_boundary = y_boundary.to(device)
h_colloc = h_colloc.to(device)
v_colloc = v_colloc.to(device)

h_colloc.requires_grad_(True)
v_colloc.requires_grad_(True)

In [ ]:
# Training
model = AirbrakePINN().to(device)
model.train()
losses = {
    "total": [],
    "data": [],
    "boundary": [],
    "physics": []
}


def train_step():
    # Flight data
    outputs_data = model(h_data, v_data)
    loss_data = torch.mean((outputs_data - y_data)**2)

    # Boundary points loss
    outputs_boundary = model(h_boundary, v_boundary)
    loss_boundary = torch.mean((outputs_boundary - y_boundary)**2)

    # Collocation points loss
    outputs_colloc = model(h_colloc, v_colloc)
    dh_dx_norm = torch.autograd.grad(outputs_colloc, h_colloc, grad_outputs=torch.ones_like(outputs_colloc), create_graph=True)[0]
    dh_dv_norm = torch.autograd.grad(outputs_colloc, v_colloc, grad_outputs=torch.ones_like(outputs_colloc), create_graph=True)[0]
    dh_dx_real = dh_dx_norm
    dh_dv_real = dh_dv_norm * (max_altitude / max_velocity)
    h_real = h_colloc * max_altitude
    v_real = v_colloc * max_velocity
    a_real = compute_acc(h_real, v_real)
    residual = (dh_dx_real * v_real) + (dh_dv_real * a_real)
    loss_physics = torch.mean(residual**2)

    # Total loss
    total_loss = (w_data * loss_data) + (w_boundary * loss_boundary) + (w_physics * loss_physics)
    total_loss.backward()

    losses["total"].append(total_loss.item())
    losses["data"].append(loss_data.item())
    losses["boundary"].append(loss_boundary.item())
    losses["physics"].append(loss_physics.item())

    return total_loss


def print_train_status(epoch, n_epochs, optimizer_name):
    if (epoch + 1) % epochs_print_rate == 0:
        print(f"[{optimizer_name}] Epoch {epoch+1}/{n_epochs} | Total: {losses['total'][-1]:.6f} | "
              f"Data: {losses['data'][-1]:.6f} | "
              f"Physics: {losses['physics'][-1]:.6f} | "
              f"Boundary: {losses['boundary'][-1]:.6f}"
              )

In [ ]:
adam_optim = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(n_epochs_adam):
    adam_optim.zero_grad()
    train_step()
    adam_optim.step()

    print_train_status(epoch, n_epochs_adam, "Adam")

In [ ]:
lbfgs_optim = torch.optim.LBFGS(model.parameters(), lr=0.1, max_iter=n_epochs_lbfgs, history_size=n_history_size_lbfgs)
lbfgs_iter = 0

def closure():
    global lbfgs_iter
    
    lbfgs_optim.zero_grad()
    loss = train_step()

    lbfgs_iter += 1

    print_train_status(lbfgs_iter, n_epochs_lbfgs, "L-BFGS")
    
    return loss

lbfgs_optim.step(closure)

In [ ]:
# Save model
total_params = sum(p.numel() for p in model.parameters())
print(f"Saving model. Total model parameters: {total_params}")

torch.save(model.state_dict(), os.path.join(os.path.abspath(''), "airbrake_pinn.pt"))

In [ ]:
# Get OR data for inference
df = pd.read_csv(os.path.join(os.path.abspath(''), "..", "..", "..", "..", "..", "sim", "hub", "data", "OR_TAIPAN.csv"))
apogee_df = df['Altitude (m)'].max()
df['y'] = apogee_df
df = df.iloc[2600:6165]
t_infer = df['# Time (s)'].values
h_infer = df['Altitude (m)'].values
v_infer = df['Vertical velocity (m/s)'].values
y_infer = df['y'].values

plt.figure(figsize=(10, 5))
plt.plot(t_infer, h_infer)
plt.show()

In [ ]:
# Simple inference and plot
model.eval()

with torch.no_grad():
    h_infer_tensor = torch.tensor(h_infer / max_altitude, dtype=torch.float32).unsqueeze(1).to(device)
    v_infer_tensor = torch.tensor(v_infer / max_velocity, dtype=torch.float32).unsqueeze(1).to(device)
    y_pred_tensor = model(h_infer_tensor, v_infer_tensor)

y_pred = y_pred_tensor.cpu().numpy()[:, 0] * max_altitude

y_pred_physics = []
for i in range(len(y_pred)):
    a = h_infer[i] + c_m / (2 * 0.5 * c_rho * c_Cd * c_A) * np.log(1.0 + (0.5 * c_rho * c_Cd * c_A * v_infer[i]**2) / (c_m * c_g))
    y_pred_physics.append(a)
y_pred_physics = np.array(y_pred_physics)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t_infer, h_infer, label="altitude (normalized)")
plt.plot(t_infer, y_infer, label="true apogee")
plt.plot(t_infer, y_pred_physics, label="true apogee (physics)")
plt.plot(t_infer, y_pred, label="predicted apogee")
plt.title("Airbrake PINN Inference")
plt.xlabel("Sample")
plt.ylabel("Normalized apogee")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
err_pred_physics = y_pred - y_pred_physics

plt.figure(figsize=(10, 5))
plt.plot(t_infer, err_pred_physics)
plt.show()

In [ ]:
err_true = y_pred - y_infer

plt.figure(figsize=(10, 5))
plt.plot(t_infer, err_true)
plt.show()

In [ ]:
err_physics = y_pred_physics - y_infer

plt.figure(figsize=(10, 5))
plt.plot(t_infer, err_physics)
plt.show()

In [ ]:
# Generate C code

def export_weights_to_c(pt_file_path, header_file_path):
    # Load the PyTorch model state dictionary
    # Use map_location='cpu' in case it was saved on a GPU
    state_dict = torch.load(pt_file_path, map_location='cpu')
    
    with open(header_file_path, "w") as f:
        f.write("#ifndef _MODEL_WEIGHTS_H\n")
        f.write("#define _MODEL_WEIGHTS_H\n\n")
        f.write("// Auto-generated weights from PyTorch model\n\n")
        
        f.write(f"#define MAX_ALTITUDE {max_altitude:.1f}f\n")
        f.write(f"#define MAX_VELOCITY {max_velocity:.1f}f\n\n")
        
        for name, tensor in state_dict.items():
            # Clean up the variable name for C/C++ (replace dots with underscores)            
            c_name = name.replace('.', '_')
            
            # Convert to numpy and flatten to a 1D array
            flat_data = tensor.detach().numpy().flatten()
            
            # Write C array declaration
            f.write(f"const float {c_name}[{len(flat_data)}] = {{\n    ")

            # Format numbers safely as C floats
            values_str = ", ".join([f"{val:.7f}f" for val in flat_data])

            # Save
            f.write(values_str)    
            f.write("\n};\n")
            
            if len(tensor.shape) == 1:
                def_name = f"SIZE_{c_name.upper()}"
                f.write(f"#define {def_name} {len(flat_data)}\n")            

            f.write("\n")

        f.write("#endif // _MODEL_WEIGHTS_H\n")

export_weights_to_c('airbrake_pinn.pt', './generated/model_weights.h')